# Cohort Definition and Prediction Target

This notebook defines the patient cohort, creates the in-hospital mortality target, and examines relationships among the patients, admissions, and ICU stays tables.

In [9]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/mimic-iv-clinical-database-demo-2.2")
HOSP_DIR = DATA_DIR / "hosp"
ICU_DIR = DATA_DIR / "icu"

patients = pd.read_csv(HOSP_DIR / "patients.csv.gz")
admissions = pd.read_csv(HOSP_DIR / "admissions.csv.gz")
icustays = pd.read_csv(ICU_DIR / "icustays.csv.gz")

print("Patients:", patients.shape)
print("Admissions:", admissions.shape)
print("ICU stays:", icustays.shape)

Patients: (100, 6)
Admissions: (275, 16)
ICU stays: (140, 8)


## Table Keys

- `subject_id` identifies a patient.
- `hadm_id` identifies a hospital admission.
- `stay_id` identifies an ICU stay.

In [11]:
patients[["subject_id"]].head()

,subject_id
0,10014729
1,10003400
2,10002428
3,10032725
4,10027445


In [6]:
admissions[["subject_id", "hadm_id"]].head()

,subject_id,hadm_id
0,10004235,24181354
1,10009628,25926192
2,10018081,23983182
3,10006053,22942076
4,10031404,21606243


In [7]:
icustays[["subject_id", "hadm_id", "stay_id"]].head()

,subject_id,hadm_id,stay_id
0,10018328,23786647,31269608
1,10020187,24104168,37509585
2,10020187,26842957,32554129
3,10012853,27882036,31338022
4,10020740,25826145,32145159


In [8]:
print("Unique patients:", patients["subject_id"].nunique())
print("Patients with admissions:", admissions["subject_id"].nunique())
print("Hospital admissions:", admissions["hadm_id"].nunique())
print("ICU stays:", icustays["stay_id"].nunique())

Unique patients: 100
Patients with admissions: 100
Hospital admissions: 275
ICU stays: 140


A patient may have multiple hospital admissions, and one hospital admission may include more than one ICU stay.

## Prediction Target

The project aims to predict in-hospital mortality among hospital admissions that include an ICU stay.

The target variable is `hospital_expire_flag`:

- `0`: the patient survived the hospital admission
- `1`: the patient died during the hospital admission

In [12]:
target_columns = [
    "subject_id",
    "hadm_id",
    "admittime",
    "dischtime",
    "deathtime",
    "hospital_expire_flag",
]

admissions[target_columns].head(10)

,subject_id,hadm_id,admittime,dischtime,deathtime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,0
5,10005817,20626031,2132-12-12 01:43:00,2132-12-20 15:04:00,NaN,0
6,10019385,20297618,2180-02-15 20:28:00,2180-02-25 13:45:00,NaN,0
7,10002495,24982426,2141-05-22 20:17:00,2141-05-29 17:41:00,NaN,0
8,10038081,20755971,2115-09-27 20:40:00,2115-10-12 00:00:00,2115-10-12 22:20:00,1
9,10019917,22585261,2182-01-07 23:25:00,2182-01-10 16:52:00,NaN,0


In [13]:
admissions["hospital_expire_flag"].value_counts()

hospital_expire_flag
0    260
1     15
Name: count, dtype: int64

In [ ]:
target_distribution = (
    admissions["hospital_expire_flag"]
    .value_counts(normalize=True) ###
    .rename(index={0: "Survived", 1: "Died"})  ###
    .mul(100)
    .round(2)
)

target_distribution

hospital_expire_flag
Survived    94.55
Died         5.45
Name: proportion, dtype: float64

## Unit of Analysis

The model will use one row per hospital admission.

Some hospital admissions contain multiple ICU stays. For this first version, only the first ICU stay within each hospital admission will be retained.

In [16]:
icustays["intime"] = pd.to_datetime(
    icustays["intime"],
    errors="coerce",
)

In [17]:
icu_stays_per_admission = (
    icustays.groupby("hadm_id")["stay_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Maximum ICU stays in one admission:", icu_stays_per_admission.max())
print(
    "Admissions with multiple ICU stays:",
    (icu_stays_per_admission > 1).sum(),
)

Maximum ICU stays in one admission: 4
Admissions with multiple ICU stays: 9


In [18]:
first_icu_stay = (
    icustays
    .sort_values(["hadm_id", "intime"])
    .drop_duplicates(subset="hadm_id", keep="first")
    .copy()
)

In [19]:
print("All ICU stays:", icustays.shape)
print("First ICU stay per admission:", first_icu_stay.shape)
print(
    "Unique admissions in first_icu_stay:",
    first_icu_stay["hadm_id"].nunique(),
)

All ICU stays: (140, 8)
First ICU stay per admission: (128, 8)
Unique admissions in first_icu_stay: 128


In [20]:
assert first_icu_stay["hadm_id"].is_unique

print("Each hospital admission now has one ICU record.")

Each hospital admission now has one ICU record.


## Merge ICU and Hospital Admission Data

The first ICU stay is merged with the hospital admission table using both `subject_id` and `hadm_id`.

In [22]:
cohort = first_icu_stay.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="inner",
    validate="one_to_one",
    suffixes=("_icu", "_admission"),
)

In [23]:
print("Cohort after ICU-admission merge:", cohort.shape)

cohort.head()

Cohort after ICU-admission merge: (128, 22)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10023771,20044587,33177122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2113-08-25 09:32:41,2113-08-27 16:27:53,2.288333,2113-08-25 07:15:00,2113-08-30 14:15:00,...,P47E1G,PHYSICIAN REFERRAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,NaN,NaN,0
1,10005909,20199380,36496303,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2144-10-29 23:09:03,2144-11-02 15:24:29,3.677384,2144-10-28 23:20:00,2144-11-02 15:23:00,...,P43BTJ,EMERGENCY ROOM,HOME,Other,ENGLISH,MARRIED,WHITE,2144-10-28 18:29:00,2144-10-29 00:10:00,0
2,10003400,20214994,32128372,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,2137-02-24 10:00:00,2137-03-19 15:45:00,...,P60ZCO,TRANSFER FROM SKILLED NURSING FACILITY,CHRONIC/LONG TERM ACUTE CARE,Medicare,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,NaN,NaN,0
3,10008454,20291550,31959184,Trauma SICU (TSICU),Trauma SICU (TSICU),2110-11-30 17:11:36,2110-12-05 16:48:24,4.983889,2110-11-30 06:31:00,2110-12-10 15:53:00,...,P77BSD,EMERGENCY ROOM,HOME HEALTH CARE,Other,ENGLISH,SINGLE,WHITE,2110-11-30 04:45:00,2110-11-30 08:03:00,0
4,10019385,20297618,39268883,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2180-02-21 08:34:06,2180-02-22 16:05:14,1.313287,2180-02-15 20:28:00,2180-02-25 13:45:00,...,P536JC,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,0


## Add Patient Demographics

Patient-level demographic information is added by merging the cohort with the patient table using `subject_id`.

In [24]:
cohort = cohort.merge(
    patients,
    on="subject_id",
    how="left",
    validate="many_to_one",
)

In [25]:
print("Final merged cohort:", cohort.shape)

cohort.head()

Final merged cohort: (128, 27)


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,marital_status,race,edregtime,edouttime,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10023771,20044587,33177122,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2113-08-25 09:32:41,2113-08-27 16:27:53,2.288333,2113-08-25 07:15:00,2113-08-30 14:15:00,...,MARRIED,WHITE,NaN,NaN,0,M,70,2113,2011 - 2013,NaN
1,10005909,20199380,36496303,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2144-10-29 23:09:03,2144-11-02 15:24:29,3.677384,2144-10-28 23:20:00,2144-11-02 15:23:00,...,MARRIED,WHITE,2144-10-28 18:29:00,2144-10-29 00:10:00,0,F,40,2144,2014 - 2016,NaN
2,10003400,20214994,32128372,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,2137-02-24 10:00:00,2137-03-19 15:45:00,...,MARRIED,BLACK/AFRICAN AMERICAN,NaN,NaN,0,F,72,2134,2011 - 2013,2137-09-02
3,10008454,20291550,31959184,Trauma SICU (TSICU),Trauma SICU (TSICU),2110-11-30 17:11:36,2110-12-05 16:48:24,4.983889,2110-11-30 06:31:00,2110-12-10 15:53:00,...,SINGLE,WHITE,2110-11-30 04:45:00,2110-11-30 08:03:00,0,F,26,2110,2011 - 2013,NaN
4,10019385,20297618,39268883,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2180-02-21 08:34:06,2180-02-22 16:05:14,1.313287,2180-02-15 20:28:00,2180-02-25 13:45:00,...,MARRIED,WHITE,NaN,NaN,0,M,44,2180,2014 - 2016,NaN


In [26]:
print("Rows in cohort:", len(cohort))
print("Unique hospital admissions:", cohort["hadm_id"].nunique())
print("Unique ICU stays:", cohort["stay_id"].nunique())
print("Unique patients:", cohort["subject_id"].nunique())

Rows in cohort: 128
Unique hospital admissions: 128
Unique ICU stays: 128
Unique patients: 100


In [27]:
assert cohort["hadm_id"].is_unique
assert cohort["stay_id"].is_unique
assert cohort["hospital_expire_flag"].isin([0, 1]).all()

print("Cohort validation passed.")

Cohort validation passed.


## Mortality Distribution in the ICU Cohort

The mortality distribution is now evaluated only among hospital admissions containing an ICU stay.

In [28]:
cohort["hospital_expire_flag"].value_counts()

hospital_expire_flag
0    113
1     15
Name: count, dtype: int64

In [29]:
icu_target_distribution = (
    cohort["hospital_expire_flag"]
    .value_counts(normalize=True)
    .rename(index={0: "Survived", 1: "Died"})
    .mul(100)
    .round(2)
)

icu_target_distribution

hospital_expire_flag
Survived    88.28
Died        11.72
Name: proportion, dtype: float64

In [30]:
mortality_summary = pd.DataFrame(
    {
        "count": cohort["hospital_expire_flag"].value_counts(),
        "percentage": (
            cohort["hospital_expire_flag"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        ),
    }
).rename(index={0: "Survived", 1: "Died"})

mortality_summary

,count,percentage
hospital_expire_flag,,
Survived,113,88.28
Died,15,11.72


## Data Leakage

The following variables will not be used as model predictors:

- `hospital_expire_flag`: this is the target variable
- `deathtime`: directly reveals that the patient died
- `dod`: directly reveals death
- `dischtime`: only known at the end of hospitalization
- `outtime`: only known after the ICU stay is complete
- `los`: completed ICU length of stay and therefore future information

Using these variables would allow the model to access information that would not be available at the intended prediction time.

## Initial Modeling Dataset

The initial modeling dataset includes patient identifiers for tracking, baseline demographic information, admission characteristics, ICU type, and the mortality target.

Identifier columns will not be used as model inputs.

In [31]:
selected_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "gender",
    "anchor_age",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "first_careunit",
    "hospital_expire_flag",
]

modeling_df = cohort[selected_columns].copy()

In [32]:
modeling_df.head()

,subject_id,hadm_id,stay_id,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag
0,10023771,20044587,33177122,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
1,10005909,20199380,36496303,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
2,10003400,20214994,32128372,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0
3,10008454,20291550,31959184,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU),0
4,10019385,20297618,39268883,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0


In [33]:
print("Modeling dataset shape:", modeling_df.shape)

Modeling dataset shape: (128, 12)


In [34]:
modeling_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   subject_id            128 non-null    int64
 1   hadm_id               128 non-null    int64
 2   stay_id               128 non-null    int64
 3   gender                128 non-null    str  
 4   anchor_age            128 non-null    int64
 5   admission_type        128 non-null    str  
 6   admission_location    128 non-null    str  
 7   insurance             128 non-null    str  
 8   marital_status        118 non-null    str  
 9   race                  128 non-null    str  
 10  first_careunit        128 non-null    str  
 11  hospital_expire_flag  128 non-null    int64
dtypes: int64(5), str(7)
memory usage: 12.1 KB


In [35]:
modeling_df.isna().sum().sort_values(ascending=False)

marital_status          10
subject_id               0
stay_id                  0
hadm_id                  0
gender                   0
anchor_age               0
admission_location       0
admission_type           0
insurance                0
race                     0
first_careunit           0
hospital_expire_flag     0
dtype: int64

In [36]:
missing_summary = pd.DataFrame(
    {
        "missing_count": modeling_df.isna().sum(),
        "missing_percentage": (
            modeling_df.isna().mean()
            .mul(100)
            .round(2)
        ),
    }
).sort_values("missing_percentage", ascending=False)

missing_summary

,missing_count,missing_percentage
marital_status,10,7.81
subject_id,0,0.00
stay_id,0,0.00
hadm_id,0,0.00
gender,0,0.00
anchor_age,0,0.00
admission_location,0,0.00
admission_type,0,0.00
insurance,0,0.00
race,0,0.00


In [37]:
categorical_columns = [
    "gender",
    "admission_type",
    "admission_location",
    "insurance",
    "marital_status",
    "race",
    "first_careunit",
]

for column in categorical_columns:
    print(f"\n{column}")
    print(modeling_df[column].value_counts(dropna=False))


gender
gender
M    69
F    59
Name: count, dtype: int64

admission_type
admission_type
EW EMER.                       62
URGENT                         25
OBSERVATION ADMIT              17
SURGICAL SAME DAY ADMISSION    15
ELECTIVE                        5
DIRECT EMER.                    4
Name: count, dtype: int64

admission_location
admission_location
EMERGENCY ROOM                            63
TRANSFER FROM HOSPITAL                    28
PHYSICIAN REFERRAL                        26
PROCEDURE SITE                             3
CLINIC REFERRAL                            3
TRANSFER FROM SKILLED NURSING FACILITY     2
PACU                                       2
INFORMATION NOT AVAILABLE                  1
Name: count, dtype: int64

insurance
insurance
Other       68
Medicare    48
Medicaid    12
Name: count, dtype: int64

marital_status
marital_status
MARRIED     47
SINGLE      42
WIDOWED     21
NaN         10
DIVORCED     8
Name: count, dtype: int64

race
race
WHITE                 

In [39]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
output_path = PROCESSED_DIR / "icu_mortality_cohort_demo.csv"

modeling_df.to_csv(
    output_path,
    index=False,
)

print(f"Saved modeling dataset to: {output_path}")

Saved modeling dataset to: ..\data\processed\icu_mortality_cohort_demo.csv


In [41]:
print("File exists:", output_path.exists())

File exists: True


## Day 2 Summary

- Prediction task: in-hospital mortality prediction
- Population: hospital admissions containing at least one ICU stay
- Unit of analysis: one hospital admission
- ICU selection: first ICU stay within each hospital admission
- Target: `hospital_expire_flag`
- Data sources: `patients`, `admissions`, and `icustays`
- Leakage variables were identified and excluded from future model inputs
- An initial modeling dataset was created for feature engineering